In [1]:
import torch

from metarcwa import Source, Lattice, IsotropicMedium, Layer, Stack, Model
from metarcwa.model.adapters import from_dispertorch, from_metashapes

from dispertorch import ConstantEps, material

from metashapes.shape import Rectangle


In [2]:
# Build the model containing the physial description of the structure using
# MetaRCWA

def build_model(
        particle_side_length_nm: torch.Tensor,
        planarization_layer_thickness_nm: torch.Tensor,
        patterned_layer_thickness_nm: torch.Tensor,
        lattice_period_nm: torch.Tensor,
        wavelength_nm: torch.Tensor,
        theta_rad: torch.Tensor,
        phi_rad: torch.Tensor,
        incidence_material: material,
        planarization_material: material,
        pattern_background_material: material,
        particle_material: material,
        transmission_material: material,
        ) -> Model:
    """
    Construct a periodic square-particle MetaRCWA model. 

    The complete layer stack from incidence side to the 
    transmission side:
    1. Semi-infinite incidence medium
    2. Homogeneous planarisation layer
    3. Patterned layer containing the square particle and
    its background which in this case is the same material 
    as the planarisation layer. 
    4. Semi-infinite transmission medium.

    Paramters:
    ----------
    particle_side_length_nm: torch.Tensor
        Side length of the square particle in a nanometers. The 
        same length is used in the x and y directions.
    planarization_layer_thickness_nm: torch.Tensor
        Thickness of the planarization layer above the patterned layer
        and below the incidence medium in nanometers.
    patterned_layer_thickness_nm: torch.Tensor
        Thickness of the layer containing the square particle in nm.
    lattice_period_nm: torch.Tensor
        Period of the square lattice in nm. Same in the x and y directions.
    wavelength_nm: torch.Tensor
        Free-space illumination wavelength(s) in nm.
    theta_rad: torch.Tensor
        Polar incidence angle(s) in radians, measured from the surface normal.
        Use zero for normal incidence.
    phi_rad: torch.Tensor
        Azimuthal incidence(s) in radians.
    incidence_material: DispersionModel
        A DispersionModel for the semi-infinite incidence medium.
    planarization_material: DispersionModel
        A DisperTorch Model for the homogeneous planarization layer
    pattern_background_material: DispersionModel
        A DisperTorch Model for the material surrounding the particle.
        This material occupies the region where the geometry mask is equal to zero.
    transmission_perm_material: DispersionModel
        A DisperTorch Model for the semi-infinite transmission/susbtrate material.
    
    DispersionModel being different models in the DisperTorch repository
    which calculates the permittitivies of the material differently. 
    They all recieve the same input, a wavelength tensor, and return the 
    same complex permittivity tensor. 

    Examples of the DispersionModel:
    - ConstantEps
    - ConstantIndex
    - Sellmeier
    - Lorentz
    - Drude
    - Composite

    Returns
    -------
    Model:
        A MetaRCWA model containing the completed stack and source. The model can 
        be resolved into a ModelSpec and translated by S4 or FMMax backend.
    """

    # Define lattice vectors
    lattice = Lattice.rectangular(px = lattice_period_nm, py = lattice_period_nm)

    # Define materials
    incidence_material = IsotropicMedium(from_dispertorch(incidence_material))
    planarization_material = IsotropicMedium(from_dispertorch(planarization_material))
    pattern_background_material = IsotropicMedium(from_dispertorch(pattern_background_material))
    transmission_material = IsotropicMedium(from_dispertorch(transmission_material))
    particle_material=IsotropicMedium(from_dispertorch(particle_material))

    # Define square geometry
    center = torch.tensor([lattice_period_nm / 2, lattice_period_nm / 2])
    size = torch.tensor([particle_side_length_nm, particle_side_length_nm])
    angle=torch.tensor(0.0)
    corner_radius = torch.tensor(0.0)
    square_geometry = Rectangle(center = center,
                                size=size,
                                angle=angle,
                                corner_radius = corner_radius
                                )

    # Define finite layers
    planarization_layer = Layer(medium_solid = planarization_material,
                                thickness=planarization_layer_thickness_nm
                                )
    patterned_layer = Layer(medium_solid=particle_material,
                            thickness=patterned_layer_thickness_nm,
                            medium_void=pattern_background_material,
                            shape_fn=square_geometry
                            )

    # Stack the layers
    stack = Stack(incidence=incidence_material,
                  layers = [planarization_layer, patterned_layer],
                  transmission=transmission_material,
                  lattice=lattice
                  )

    # Define the source
    source = Source(wavelength=wavelength_nm,
                    theta=theta_rad,
                    phi=phi_rad
                    )

    return Model(stack, source)

In [3]:
from dataclasses import dataclass

@dataclass
class S4Config:
    """Numerical settings used directly by S4

    Attributes
    ----------
    num_basis: int
    lattice_truncation: str
    """

    # Set defaults
    num_basis: int = 25
    lattice_truncation: str = 'Circular' 

    def __post_init__(self) -> None:
        """Check that the supplied settings are valid."""

        if self.num_basis <=0:
            raise ValueError("num_basis must be positive")

        allowed_truncations = {
            "Circular",
            "Parallelogramic"
        }

        if self.lattice_truncation not in allowed_truncations:
            raise ValueError(
                "lattice_truncation must be "
                "'Circular' or 'Parallelogramic'"
            )

In [4]:
# S4